# ⚡ AI-Powered Energy Consumption Forecasting
### Exploratory Data Analysis & Interactive Demo

This notebook walks through the full project pipeline interactively:
1. Load and explore the dataset
2. Visualise consumption patterns
3. Build features
4. Train and evaluate the model
5. Generate a 24-hour forecast

**Dataset:** UCI Household Electric Power Consumption (or synthetic fallback)

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '..')  # allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 4)
sns.set_style('whitegrid')
print('Libraries loaded ✓')

## 1. Generate / Load Dataset

In [ ]:
# Generate synthetic dataset (run once)
os.chdir('..')   # move to project root
from src.generate_data import generate_synthetic_data
df_raw = generate_synthetic_data(years=4)
df_raw.head(10)

In [ ]:
# Dataset info
print('Shape      :', df_raw.shape)
print('Date range :', df_raw.index[0], '→', df_raw.index[-1])
print('Missing    :', df_raw.isnull().sum().values[0])
df_raw.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Full time-series overview
fig, ax = plt.subplots(figsize=(15, 3))
df_raw['power_kw'].plot(ax=ax, lw=0.4, color='#2563EB', alpha=0.7)
ax.set_title('Household Energy Consumption — Full Dataset')
ax.set_ylabel('Power (kWh)')
plt.tight_layout()

In [ ]:
# One typical week (2020-W04)
week = df_raw['2020-01-20':'2020-01-26']
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(week.index, week['power_kw'], color='#2563EB', lw=1.5)
ax.fill_between(week.index, week['power_kw'], alpha=0.1, color='#2563EB')
ax.set_title('Typical Week — Hourly Consumption')
ax.set_ylabel('Power (kWh)')
plt.tight_layout()

In [ ]:
# Average by hour
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

hourly = df_raw.groupby(df_raw.index.hour)['power_kw'].mean()
axes[0].bar(hourly.index, hourly.values, color='#2563EB', alpha=0.85)
axes[0].set_title('By Hour of Day')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Avg kWh')

days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
daily = df_raw.groupby(df_raw.index.dayofweek)['power_kw'].mean()
colors = ['#D97706' if i >= 5 else '#16A34A' for i in range(7)]
axes[1].bar(days, daily.values, color=colors, alpha=0.85)
axes[1].set_title('By Day of Week')

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df_raw.groupby(df_raw.index.month)['power_kw'].mean()
axes[2].plot(months, monthly.values, marker='o', color='#7C3AED', lw=2)
axes[2].fill_between(range(12), monthly.values, alpha=0.12, color='#7C3AED')
axes[2].set_title('By Month')
axes[2].set_xticklabels(months, rotation=45, fontsize=8)
axes[2].set_xticks(range(12))

plt.suptitle('Consumption Patterns', y=1.02)
plt.tight_layout()

## 3. Preprocessing & Feature Engineering

In [ ]:
from src.preprocess import load_and_clean
from src.features   import build_features

df   = load_and_clean()
X, y, feature_cols = build_features(df)

print('Feature matrix shape:', X.shape)
X.head()

In [ ]:
# Correlation heatmap
import matplotlib
corr = X.assign(power_kw=y).corr()
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()

## 4. Model Training & Evaluation

In [ ]:
from src.model    import train_model
from src.evaluate import evaluate_model

model, X_train, X_test, y_train, y_test = train_model(X, y)
y_pred, fi_sorted = evaluate_model(model, X_test, y_test, feature_cols)

In [ ]:
# Visual: actual vs predicted (2 weeks)
n = 14 * 24
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_test.values[:n],  label='Actual',    color='#2563EB', lw=1.3)
ax.plot(y_pred[:n],         label='Predicted', color='#16A34A', lw=1.3, linestyle='--')
ax.set_title('Actual vs Predicted — 2-Week Test Window')
ax.set_ylabel('Power (kWh)')
ax.set_xlabel('Hours')
ax.legend()
plt.tight_layout()

In [ ]:
# Feature importances
fi = fi_sorted.sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind='barh', ax=ax, color='#7C3AED', alpha=0.85)
ax.set_title('Feature Importance — Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()

## 5. 24-Hour Forecast

In [ ]:
from src.forecast import forecast_future

forecast_df = forecast_future(model, df, feature_cols, hours=24)
forecast_df.head()

In [ ]:
# Forecast chart
import matplotlib.dates as mdates

t   = forecast_df.index
val = forecast_df['predicted_kw']
std = val.std()

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(t, val - std, val + std, alpha=0.15, color='#2563EB', label='±1 std dev')
ax.plot(t, val, color='#2563EB', lw=2, marker='o', ms=5, label='Forecast')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d %b'))
ax.set_title('24-Hour Energy Demand Forecast')
ax.set_ylabel('Power (kWh)')
ax.legend()
plt.tight_layout()

print(f'Peak demand: {val.max():.3f} kWh at {val.idxmax().strftime("%H:%M %d %b")}')
print(f'Min  demand: {val.min():.3f} kWh at {val.idxmin().strftime("%H:%M %d %b")}')

## 6. Save All Charts

In [ ]:
from src.visualize import (
    plot_actual_vs_predicted,
    plot_feature_importance,
    plot_residuals,
    plot_forecast,
    plot_eda_overview,
)

plot_eda_overview(df)
plot_actual_vs_predicted(y_test, y_pred)
plot_feature_importance(fi_sorted)
plot_residuals(y_test, y_pred)
plot_forecast(forecast_df)

print('\nAll charts saved to outputs/images/')

---
## Summary

| Metric | Value |
|--------|-------|
| Model  | Random Forest Regressor |
| Features | 9 (lag, rolling, temporal) |
| Train size | 80% chronological |
| Test size | 20% chronological |
| Forecast horizon | 24 hours |

Run `python main.py --hours 168` for a 7-day forecast.